# SRQ-FLY P2B — final three-dataset backend confirmation
This notebook restores the immutable train-only choices from the earlier self-contained study and compares Exact FLY, optimized SRQ-FLY P2B, and raw-feature Ridge over six paired replicates on CIFAR-100, CUB-200-2011, and the disclosed legacy ImageNet-R split. The test splits were consumed by an earlier SRQ implementation; therefore this is an optimized-backend confirmation, not a fresh first-use held-out study. Run cells strictly from top to bottom.

In [ ]:
# Edit repository/path values only. Do not edit seeds, lambdas, methods, or backend settings.
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
SELECTION_ARTIFACT = '/content/srq_fly_selfcontained_three_dataset_results.zip'
FEATURE_CACHE_ROOT = '/content/srq_p2b_confirmation_features'
FINAL_WTA_ROOT = '/content/srq_p2b_confirmation_wta'
SELECTION_ROOT = '/content/srq_p2b_locked_selection'
OUTPUT_ROOT = '/content/srq_p2b_confirmation_results'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = 'c5716da80ee732b6456d69b57175b999715c97efa9b9e8f6ab464be94a9211f3'
EXPECTED_RUNNER_SHA256 = '98f8dbc5f46de87f7066ef8afdf00a83b3fafb4f056d142bf25ae14cacb3839f'
EXPECTED_SELECTION_ZIP_SHA256 = 'e4b630781ff6f69deaecb63dda9926d256cd6b654ef4b51a682bf3ef94e6490b'

In [ ]:
# Fresh clone, dependencies, GPU, and immutable source identities.
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG = 'configs/srq_fly_p2b_final_confirmation.json'
RUNNER = 'tools/srq_fly_p2b_final_confirmation.py'
BASE_PROTOCOL = 'configs/srq_fly_selfcontained_final.json'
BASE_RUNNER = 'tools/srq_fly_selfcontained.py'
assert sha(CONFIG) == EXPECTED_CONFIG_SHA256
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must start clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('P2B CONFIG/RUNNER IDENTITY: PASS')

In [ ]:
# Restore only the three immutable train-only selection files from the prior evidence ZIP.
artifact = Path(SELECTION_ARTIFACT)
assert artifact.is_file(), f'Upload {Path(SELECTION_ARTIFACT).name} with the Colab Files sidebar.'
assert zipfile.is_zipfile(artifact), 'Selection artifact is not a valid ZIP.'
assert sha(artifact) == EXPECTED_SELECTION_ZIP_SHA256, 'Selection artifact SHA-256 mismatch.'
selection_root = Path(SELECTION_ROOT)
if selection_root.exists(): shutil.rmtree(selection_root)
members = {
  'cifar100': 'train_only_selection/cifar100_selection.json',
  'cub200': 'train_only_selection/cub200_selection.json',
  'imagenetr': 'train_only_selection/imagenetr_selection.json',
}
with zipfile.ZipFile(artifact) as archive:
    for key, member in members.items():
        target = selection_root/key/'selection.json'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
from tools import srq_fly_p2b_final_confirmation as confirmation
locked_config = confirmation._read_config(Path(CONFIG))
confirmation._read_base_protocol(locked_config, Path(BASE_PROTOCOL))
selected = confirmation._validate_selections(locked_config, Path(BASE_PROTOCOL), selection_root)
print(json.dumps(selected, indent=2))
print('IMMUTABLE TRAIN-ONLY SELECTION EVIDENCE: PASS')

In [ ]:
# Download the exact frozen ViT checkpoint and processed dataset sources.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS = {
  'cifar100': kagglehub.dataset_download('zaphat206/cifar-100'),
  'cub200': kagglehub.dataset_download('zaphat206/cub-200-2011'),
  'imagenetr': kagglehub.dataset_download('zaphat206/imagenet-r'),
}
print('checkpoint:', CHECKPOINT_PATH)
print(json.dumps(DATASET_ROOTS, indent=2))

In [ ]:
# Audit dataset identity without extracting any test feature.
CUB_AUDIT = '/content/cub_p2b_confirmation_audit.json'
IMAGENETR_AUDIT = '/content/imagenetr_p2b_confirmation_audit.json'
cub = subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca'])
assert cub.returncode == 0
imagenetr = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert imagenetr.returncode == 2, 'Expected the locked legacy-overlap disclosure.'
audit = json.loads(Path(IMAGENETR_AUDIT).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET AUDIT PASS; ImageNet-R remains labelled legacy processed split.')

In [ ]:
# Extract frozen TRAIN features only. Every test.pt must remain absent here.
protocol = json.loads(Path(BASE_PROTOCOL).read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg = protocol['datasets'][key]
    cache = Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only',
          '--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,
          '--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],
          '--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_{key}',
          '--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit',
          '--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),
          '--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}', flush=True)
        subprocess.run(command, check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
    print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN-ONLY FEATURE CACHES READY')

In [ ]:
# Correctness and selection-provenance gate; no test feature is opened.
subprocess.run([sys.executable,'-m','pytest','-q',
  'tests/test_srq_fly_p2b_final_confirmation.py',
  'tests/test_srq_fly_priority2b_memory.py',
  'tests/test_srq_fly_priority2d_equivalence.py',
  'tests/test_srq_fly_selfcontained.py'], check=True)
for key in ('cifar100','cub200','imagenetr'):
    assert not (Path(FEATURE_CACHE_ROOT)/key/'test.pt').exists()
print('P2B FINAL CONFIRMATION CORRECTNESS GATE: PASS')

In [ ]:
# Lock the unchanged legacy selections before crossing the test boundary.
dirty = subprocess.check_output(['git','status','--porcelain'], text=True).strip()
assert not dirty, f'Repository source changed before lock:\n{dirty}'
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-u',BASE_RUNNER,'lock','--protocol',BASE_PROTOCOL,
  '--selection-root',SELECTION_ROOT,'--output-root',OUTPUT_ROOT,
  '--require-clean-git'], check=True)
AUTHORIZATION = str(Path(OUTPUT_ROOT)/'authorization.json')
authorization = json.loads(Path(AUTHORIZATION).read_text())
print(json.dumps(authorization['selected_hyperparameters'], indent=2))
subprocess.run([sys.executable,'-u',RUNNER,'lock','--config',CONFIG,
  '--base-protocol',BASE_PROTOCOL,'--selection-root',SELECTION_ROOT,
  '--authorization',AUTHORIZATION,'--output-root',OUTPUT_ROOT,
  '--require-clean-git'], check=True)
CONFIRMATION_AUTHORIZATION = str(Path(OUTPUT_ROOT)/'confirmation_authorization.json')
print(json.dumps(json.loads(Path(CONFIRMATION_AUTHORIZATION).read_text()), indent=2))
print('BASE + P2B CONFIRMATION TEST BOUNDARIES: LOCKED')

## Test boundary
The selection files, lambdas, seeds, source identities, and authorization are now hashed. From the next cell onward test features are visible. Do not change any method setting or stop based on accuracy. Interrupted units may only resume under the identical authorization and source context.

In [ ]:
# Materialize TEST features only after authorization.
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u',BASE_RUNNER,'extract-test','--protocol',BASE_PROTOCOL,
      '--dataset-key',key,'--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],
      '--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda',
      '--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'TEST EXTRACTION START {key}', flush=True)
    subprocess.run(command, check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Helper for one resumable six-replicate dataset confirmation.
AUDIT_PATHS = {'cifar100':None, 'cub200':CUB_AUDIT, 'imagenetr':IMAGENETR_AUDIT}
def run_dataset(key):
    command = [sys.executable,'-u',RUNNER,'evaluate','--config',CONFIG,
      '--base-protocol',BASE_PROTOCOL,'--dataset-key',key,
      '--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--confirmation-authorization',CONFIRMATION_AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),
      '--code-cache-root',FINAL_WTA_ROOT,'--output-root',OUTPUT_ROOT,
      '--device','cuda']
    if AUDIT_PATHS[key]: command += ['--dataset-audit',AUDIT_PATHS[key]]
    print(f'FINAL CONFIRMATION START {key}: 6 paired replicates x 3 methods', flush=True)
    subprocess.run(command, check=True)
    result = json.loads(Path(OUTPUT_ROOT,key,'confirmation_results.json').read_text())
    assert result['status'] == 'CONFIRMATION_COMPLETE'
    print(f'FINAL CONFIRMATION COMPLETE {key}')

In [ ]:
# CIFAR-100 confirmation. Safe to rerun after interruption.
run_dataset('cifar100')

In [ ]:
# CUB-200-2011 confirmation. Safe to rerun after interruption.
run_dataset('cub200')

In [ ]:
# Legacy processed ImageNet-R confirmation. Safe to rerun after interruption.
run_dataset('imagenetr')

In [ ]:
# Aggregate mean, sample standard deviation, paired 95% CI, and compact tables.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--config',CONFIG,
  '--output-root',OUTPUT_ROOT], check=True)
import pandas as pd
summary = json.loads(Path(OUTPUT_ROOT,'final_confirmation_summary.json').read_text())
metrics = pd.read_csv(Path(OUTPUT_ROOT,'metrics_summary.csv'))
display(metrics[['dataset','method','final_accuracy_mean','final_accuracy_sample_std',
  'average_incremental_accuracy_mean','average_incremental_accuracy_sample_std',
  'persistent_state_bytes_mean','total_update_seconds_mean']])
print('Paired P2B - Exact FLY AIA (pp):')
print(json.dumps(summary['paired_p2b_minus_exact_fly_aia'], indent=2))
print('STATUS:', summary['status'])
print('DISCLOSURE:', summary['prior_test_use_disclosure'])

In [ ]:
# Publication-style comparison: accuracy, learner state, update time, and task curves.
import matplotlib.pyplot as plt
import numpy as np
curves = pd.read_csv(Path(OUTPUT_ROOT,'task_curves.csv'))
labels = {'exact_fly_10000':'Exact FLY-10000','srq_fly_p2b_10000':'SRQ-FLY P2B','raw_ridge':'Raw Ridge'}
colors = {'exact_fly_10000':'#4C78A8','srq_fly_p2b_10000':'#F58518','raw_ridge':'#54A24B'}
datasets = ['cifar100','cub200','imagenetr']
methods = list(labels)
fig, axes = plt.subplots(2,2,figsize=(14,10))
x = np.arange(len(datasets)); width = 0.24
for j, method in enumerate(methods):
    rows = metrics.set_index(['dataset','method']).loc[[(d,method) for d in datasets]]
    axes[0,0].bar(x+(j-1)*width, rows['average_incremental_accuracy_mean'], width,
      yerr=rows['average_incremental_accuracy_sample_std'], label=labels[method], color=colors[method], capsize=3)
    axes[0,1].bar(x+(j-1)*width, rows['final_accuracy_mean'], width,
      yerr=rows['final_accuracy_sample_std'], label=labels[method], color=colors[method], capsize=3)
    axes[1,0].bar(x+(j-1)*width, rows['persistent_state_bytes_mean']/2**20, width,
      label=labels[method], color=colors[method])
axes[0,0].set_title('Average incremental accuracy'); axes[0,0].set_ylabel('AIA (%)')
axes[0,1].set_title('Final accuracy'); axes[0,1].set_ylabel('Accuracy (%)')
axes[1,0].set_title('Persistent learner state'); axes[1,0].set_ylabel('MiB (log scale)'); axes[1,0].set_yscale('log')
for ax in axes.flat[:3]: ax.set_xticks(x, datasets); ax.grid(axis='y',alpha=.25)
for (dataset,method), group in curves.groupby(['dataset','method']):
    stats = group.groupby('task_fraction')['average_seen_accuracy'].agg(['mean','std']).reset_index()
    axes[1,1].plot(stats['task_fraction'],stats['mean'],color=colors[method],
      linestyle={'cifar100':'-','cub200':'--','imagenetr':':' }[dataset].strip(),
      alpha=.9,label=f'{dataset}: {labels[method]}')
axes[1,1].set_title('Average seen-class accuracy over stream'); axes[1,1].set_xlabel('Task fraction'); axes[1,1].set_ylabel('Accuracy (%)'); axes[1,1].grid(alpha=.25)
handles, names = axes[0,0].get_legend_handles_labels(); fig.legend(handles,names,loc='upper center',ncol=3)
fig.tight_layout(rect=(0,0,1,.95))
FIGURE_PATH = Path(OUTPUT_ROOT,'p2b_final_confirmation.png')
fig.savefig(FIGURE_PATH,dpi=200,bbox_inches='tight'); plt.show()
print('FIGURE:', FIGURE_PATH)

In [ ]:
# Export compact evidence only; feature and WTA caches are deliberately excluded.
from google.colab import files
staging = Path('/content/srq_fly_p2b_final_confirmation_export')
if staging.exists(): shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copytree(OUTPUT_ROOT, staging/'results')
shutil.copytree(SELECTION_ROOT, staging/'train_only_selection')
for source, name in [(CONFIG,'locked_confirmation_config.json'),
                     (BASE_PROTOCOL,'locked_base_protocol.json'),
                     (RUNNER,'locked_confirmation_runner.py'),
                     (BASE_RUNNER,'locked_base_runner.py'),
                     (CUB_AUDIT,'cub_dataset_audit.json'),
                     (IMAGENETR_AUDIT,'imagenetr_dataset_audit.json')]:
    shutil.copy2(source, staging/name)
(staging/'repo_commit.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()+'\n')
archive_base = '/content/srq_fly_p2b_final_confirmation'
archive = Path(shutil.make_archive(archive_base,'zip',staging.parent,staging.name))
print('ZIP:',archive,'bytes=',archive.stat().st_size,'sha256=',sha(archive))
print('Feature/WTA caches excluded; they are experiment infrastructure, not learner state.')
files.download(str(archive))